In [ ]:
import os

import numpy as np
import pandas as pd
from wfdb import rdsamp
from tqdm import tqdm

from _multilabel_stratified_sampling import stratify


dataset_path = "/opt/gpudata/ecg/cinc-2020"
subset_root = "/opt/gpudata/ecg/temp"

# Make Full Dataset
The Georgia subset of the CinC dataset does not come with a metadata table, so we create one here before creating the subsets. The primary reason for this is so we can create stratified splits for our experiments without needing to do so on the fly.

In [ ]:
# metadata is stored with the samples, so first find all the samples
ignore = {
    "index.html",
    "RECORDS",
}
paths = set()
for root, _, files in os.walk(os.path.join(dataset_path, "training/georgia")):
    for f in files:
        if f in ignore:
            continue
        fname = os.path.splitext(f)[0]
        paths.add(os.path.join(root, fname))

In [ ]:
# then load all samples
data = []
waveforms = []
skipped = []
for path in tqdm(paths):
    x, meta = rdsamp(path)
    comments = dict()
    for c in meta["comments"]:
        k, v = c.split(": ")
        comments[k] = v
    filename = path.replace(dataset_path, "").lstrip(os.sep)
    # seems like there are some cases that are seemingly 5-second or
    # are otherwise incorrectly coded - skip these
    if x.shape[0] == 2500 and meta["fs"] == 500:
        skipped.append((filename, "bad length"))
        continue
    elif np.isnan(x).any():
        skipped.append((filename, "contains nans"))
        continue
    data.append({
        "frequency": meta["fs"],
        "age": float(comments["Age"]),
        "sex": comments["Sex"],
        "snomed-codes": comments["Dx"],
        "filename": filename,
    })
    waveforms.append(x)
waveforms = np.stack(waveforms)
print(f"skipped {len(skipped)} samples")

In [ ]:
# create and filter metadata table
df = pd.DataFrame(data)
assert not df["snomed-codes"].isna().any()
df = df[df["age"].notna() & df["sex"].isin({"Male", "Female"})]
df = df.sort_values("filename")
mask = df.index.to_numpy()

df = df.reset_index(drop=True)
waveforms = waveforms[mask]

df["ecg_id"] = np.arange(len(df))
df["patient_id"] = np.arange(len(df))
df = df[["patient_id", "ecg_id", "sex", "age", "snomed-codes", "frequency", "filename"]]

In [ ]:
# all 500 Hz (uniform sampling makes derivation of sample mean/std easier)
df["frequency"].value_counts()

In [ ]:
# 54:46 ratio male:female
df["sex"].value_counts()

In [ ]:
# age quartiles: [14, 51, 62, 72, 89] - right censored age
df["age"].describe()

In [ ]:
# get multilabel data
labels = df["snomed-codes"].str.get_dummies(sep=",") # one-hot encode from comma separated list of codes
label_map = pd.read_csv(os.path.join(dataset_path, "Dx_map.csv")).set_index("SNOMED CT Code")
labels.columns = [label_map.loc[int(x), "Abbreviation"] for x in labels.columns]

In [ ]:
# we need to make train/val/test so consider only labels with some minimum of examples
# a threshold of 10 was chosen heuristically based on if the stratifier could preserve
# at least 1 positive per label per split (which is difficult in the multilabel setup)
# this also follows what was done by the authors for https://arxiv.org/pdf/2509.25095
label_counts = labels.sum(axis=0)
stratifiable_labels = label_counts[label_counts >= 10].index
labels = labels[stratifiable_labels]

In [ ]:
# 50 binary labels
len(stratifiable_labels)

In [ ]:
binned_age = pd.cut(df["age"], [0, 20, 40, 60, 80, 100])
one_hot_age = pd.get_dummies(binned_age).astype(int)
one_hot_sex = pd.get_dummies(df["sex"]).astype(int)

In [ ]:
stratifier = pd.concat([one_hot_sex, one_hot_age, labels], axis=1)

In [ ]:
# no patient identifiers in this dataset, assume each ECG belongs to a unique patient
n_samples, n_classes = stratifier.shape

label_lists = [np.where(row)[0].tolist() for row in stratifier.to_numpy()]

stratified_ids, stratified_labels = stratify(
    data=label_lists,
    classes=list(range(n_classes)),
    ratios=[0.80245, 0.098775, 0.098775], # these ratios get us a perfect 8192 train split w/ the random seed
    qualities=[2] * n_samples, # no notion of quality
    ecgs_per_patient=[1] * n_samples,
    nr_clean_folds=0,
)

In [ ]:
df = pd.concat([df, labels], axis=1)
df["split"] = "no-split"
for split_idxs, split in zip(stratified_ids, ["train", "val", "test"]):
    print(f"{split}: {len(split_idxs)}")
    # check that none of the labels are empty in the split subset
    assert not (df.loc[split_idxs, stratifiable_labels].sum(axis=0) == 0).any()
    df.loc[split_idxs, "split"] = split

In [ ]:
if df["age"].apply(lambda x: x.is_integer()).all():
    df["age"] = df["age"].astype(int)

In [ ]:
df.to_csv(os.path.join(dataset_path, "georgia.csv"), index=False)

### Info for Defines

In [ ]:
# dataset task labels
stratifiable_labels.sort_values().to_list()

In [ ]:
# while we're at it, gather waveform stats from the train set
train_waveforms = waveforms[stratified_ids[0]]

In [ ]:
lowers, uppers = np.percentile(train_waveforms, [0.1, 99.9], axis=(0, 1))
display(lowers.tolist())
display(uppers.tolist())

In [ ]:
train_waveforms_clipped = np.clip(train_waveforms, lowers, uppers)
means = train_waveforms_clipped.mean(axis=(0, 1))
stds = train_waveforms_clipped.std(axis=(0, 1))
display(means.tolist())
display(stds.tolist())

# Make subsets

we'll use the same trick as in the very-small ptb-xl subsets where we use the stratifier, but we'll assign selected samples to be in the last subfold using the quality indicator, thus ensuring that all subsets will have at least 1 of each label

In [ ]:
from collections import defaultdict
from pass_pclr.defines import CINC_TARGETS

In [ ]:
df = pd.read_csv(os.path.join(dataset_path, "georgia.csv"))
train_df = df[df["split"] == "train"]
val_test_df = df[df["split"].isin(["val", "test"])]

In [ ]:
binned_age = pd.cut(train_df["age"], [0, 20, 40, 60, 80, 100])
one_hot_age = pd.get_dummies(binned_age).astype(int)
one_hot_sex = pd.get_dummies(train_df["sex"]).astype(int)
labels = train_df[CINC_TARGETS]
stratifier = pd.concat([one_hot_sex, one_hot_age, labels], axis=1)

In [ ]:
# no patient identifiers in this dataset, assume each ECG belongs to a unique patient
n_samples, n_classes = stratifier.shape

# stratify creates a patient ID based on the passed in list
# since we use a subset of the entire dataset, we need a way
# to map back to the IDs of the original whole dataset
remap = {i: v for i, v in enumerate(train_df.index)}

# to ensure the train set always has every possible label, we hijack the stratifier's
# notion of quality to ensure that the final split has those patients/ecgs
rng = np.random.default_rng(seed=42)
selected_idxs = set()
for target in CINC_TARGETS:
    # this gets us indices in the subset, not the original dataset patient IDs
    candidates = np.argwhere(labels[target]).flatten()
    idx = rng.choice(candidates)
    selected_idxs.add(idx)
assert len(selected_idxs) < 256 # should not be larger than smallest subset we aim to make
qualities = [4 if i in selected_idxs else 2 for i in range(n_samples)]

label_lists = [np.where(row)[0].tolist() for row in stratifier.to_numpy()]

stratified_ids, stratified_labels = stratify(
    data=label_lists,
    classes=list(range(n_classes)),
    ratios=[0.03125] * 32, # need to construct powers of 2 from 8192 to 256
    qualities=qualities,
    ecgs_per_patient=[1] * n_samples,
    nr_clean_folds=1,
    random_seed=2, # find a random seed that makes the last subfold precisely size 256
)

In [ ]:
# sort the subfolds by size for easier manual inspection
subset_idxs_by_size = defaultdict(list)
for i, subset_ids in enumerate(stratified_ids):
    size = len(subset_ids)
    subset_idxs_by_size[size].append(i)

In [ ]:
# select folds to use to construct target, prefer 256 sized subsets
# last fold will has the preselected samples to ensure all labels accounted for
# and is therefore present in all constructed subsets
{k: subset_idxs_by_size[k] for k in sorted(subset_idxs_by_size.keys())}

In [ ]:
def make_train_subset(subfolds: list[int], name: str):
    # remap idxs back to the original train set
    train_idxs = [remap[x] for subfold in subfolds for x in stratified_ids[subfold]]
    subset_df = pd.concat([train_df.loc[train_idxs], val_test_df])

    subset_path = os.path.join(subset_root, f"cinc-2020-{name}")
    os.makedirs(subset_path, exist_ok=True)

    subset_df.to_csv(os.path.join(subset_path, "georgia.csv"), index=False)

    # also link source waveform data
    os.symlink(
        src=os.path.join(dataset_path, "training"),
        dst=os.path.join(subset_path, "training"),
    )

In [ ]:
for name, target_size, folds in [
    ("4k", 4096, [3, 6, 8, 12, 13, 15, 16, 18, 23, 24, 25, 26, 27, 28, 30, 31]),
    ("2k", 2048, [8, 12, 13, 15, 16, 18, 26, 31]),
    ("1k", 1024, [16, 18, 26, 31]),
    ("512", 512, [26, 31]),
    ("256", 256, [31]),
]:
    idxs = [x for fold in folds for x in stratified_ids[fold]]
    assert len(set(idxs)) == target_size
    make_train_subset(folds, name)